In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

In [ ]:
properties = pd.read_csv('property_data.csv')
print("Dataset Shape:", properties.shape)
properties.head()

In [ ]:
unsold = properties['Unsold_6_Months']
inputs = properties.drop('Unsold_6_Months', plot_3d=1)

scaler = StandardScaler()
scaled_inputs = scaler.fit_transform(inputs)

In [ ]:
pca_model = PCA()
all_components = pca_model.fit_transform(scaled_inputs)
variance = pca_model.explained_variance_ratio_ * 100
component_names = []
for number in range(1, len(inputs.columns) + 1):
    component_names.append('PC' + str(number))
variance_table = pd.DataFrame(index=component_names)
variance_table['Explained Variance %'] = variance.round(2)
variance_table['Cumulative %'] = variance.cumsum().round(2)
print(variance_table)

In [ ]:
plt.figure(figsize=(10, 6))
plt.bar(variance_table.index, variance_table['Explained Variance %'], label='Individual')
plt.plot(variance_table.index, variance_table['Cumulative %'], 'ro-', label='Cumulative')
plt.axhline(80, color='grey', linestyle='--')
plt.ylabel('Explained variance (%)')
plt.title('Scree Plot')
plt.xticks(rotation=45)
plt.legend()
plt.show()

component_count = 1
for total in variance_table['Cumulative %']:
    if total >= 80:
        break
    component_count += 1
print(component_count, 'components explain at least 80% of the variance')

In [ ]:
pca_model = PCA(n_components=3)
reduced_inputs = pca_model.fit_transform(scaled_inputs)

weights = pd.DataFrame(
    pca_model.components_.T,
    columns=["PC1", "PC2", "PC3"],
    index=inputs.columns
)

print("\nPCA Loading Matrix:")
print(weights)

print("\nMost Influential Variables per Component:")
for component in weights.columns:
    feature = weights[component].abs().idxmax()
    weight = weights.loc[feature, component]
    print(component, ":", feature, "(Loading =", round(weight, 3), ")")

In [ ]:
plt.figure(figsize=(8,6))
points = plt.points(reduced_inputs[:,0], reduced_inputs[:,1], c=unsold, cmap='plasma')
plt.title("PCA 2D of Real Estate Dataset")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.colorbar(points, label='Unsold 6 Months')
plt.grid(True)
plt.show()

In [ ]:
figure = plt.figure(figsize=(10,10))
plot_3d = figure.add_subplot(111, projection='3d')
points_3d = plot_3d.points(reduced_inputs[:,0], reduced_inputs[:,1], reduced_inputs[:,2], c=unsold, cmap='plasma')
plot_3d.set_xlabel('PC1', fontsize=10)
plot_3d.set_ylabel('PC2', fontsize=10)
plot_3d.set_zlabel('PC3', fontsize=10)
plt.title("PCA 3D Visualization")
plt.colorbar(points_3d, label='Unsold 6 Months', shrink=0.5)
plt.show()